# Read Waveform Data
As the name suggests this notebook constructs the working data for this course from the raw archives store on amazon S3.  Comments are pretty scarce.  What is done here will be discussed on the last session of the course so my comments are pretty terse. 

In [1]:
from mspasspy.client import Client
mspass_client=Client()
db=mspass_client.get_database("ES2026")
dask_client = mspass_client.get_scheduler()

In [2]:
dask_client

<Client: 'tcp://127.0.0.1:8786' processes=4 threads=4, memory=0 B>

This incantation is necessary to run code in local python files on dask workers.   "upload_files" sends the python code to each worker.  A plugin is a concept that will be discussed in the class.

In [3]:
dask_client.upload_file("s3_worker_plugin.py")
dask_client.upload_file("s3segment_reader.py")

from s3_worker_plugin import S3Worker
s3worker = S3Worker()
dask_client.register_plugin(s3worker)

{'tcp://127.0.0.1:35123': {'status': 'OK'},
 'tcp://127.0.0.1:38635': {'status': 'OK'},
 'tcp://127.0.0.1:39651': {'status': 'OK'},
 'tcp://127.0.0.1:46873': {'status': 'OK'}}

This is not essential, but useful to verify the ImmortalS3Client thread is running.

Fetch only data suitable for teleseismic P wave processing for receiver functions.  Use the "delta" attribute from assoc to limit data to epicentral distances between 30 and 95.   Note in this notebook that does nothing as we already filtered input that way, but shows an alternative way to do tha with MongoDB.

In [4]:
query={"delta" : {"$gte" : 30.0, "$lte" : 95.0}}
cursor=db.arrival_css30.find(query)
doclist = list(cursor)
print("Size of arrival table loaded = ",len(doclist))  # validate this is reasonable

Size of arrival table loaded =  82848


In [5]:
# reader we use requires a DataFrame as input 
import pandas as pd
df = pd.DataFrame(doclist)
print(df)

                            _id      evid      orid      arid phase   delta  \
0      6a4808a80872497db262b3c1  298930.0  472052.0  24259456     P  70.912   
1      6a4808a80872497db262b3c2  298930.0  472052.0  24259457     P  71.152   
2      6a4808a80872497db262b3c3  298930.0  472052.0  24259458     P  71.325   
3      6a4808a80872497db262b3c4  298930.0  472052.0  24259459     P  71.991   
4      6a4808a80872497db262b3c5  298930.0  472052.0  24259460     P  72.167   
...                         ...       ...       ...       ...   ...     ...   
82843  6a4808a90872497db263fa1e  299892.0  473209.0  24384852     P  60.363   
82844  6a4808a90872497db263fa1f  299892.0  473209.0  24384853     P  60.761   
82845  6a4808a90872497db263fa20  299892.0  473209.0  24384854     P  60.834   
82846  6a4808a90872497db263fa21  299892.0  473209.0  24384855     P  61.497   
82847  6a4808a90872497db263fa22  299892.0  473209.0  24384856     P  65.421   

         seaz    esaz  timeres iphase  fm  source_l

To keep this dataset within the bounds of GeoLab quotas for this class we need to limit this further.   There are a lot of picks in the ANF data for small events that challenge RF deconvolution algorithms anyway, so we will impose a magnitude filter.   A problem is that the "arrival_css30" collection does not contain a magnitude number but only a link to the source collection.  We will use the pandas api to address this nasty problem.  This is a specialized solution for this exercise, but is hopefully informative to students.

In [6]:
# first load up a dictionary of magnitudes keyed by orid
cursor=db.source.find({})
mags = dict()
for doc in cursor:
    orid=int(doc["orid"])   # int may not be necessary but safer
    if "magnitude" in doc:
        mags[orid] = doc["magnitude"]
    else:
        mags[orid] = -99.

In [7]:
# create a numpy array to hold mag values for each df row and load it with mags values
import numpy as np
N=len(df)
mag_column = np.zeros(N)
i=0
for row in df.itertuples():
    orid = int(row.orid)
    if orid in mags:
        mag_column[i] = mags[orid]
    else:
        mag_column[i] = -99.
    i += 1
df["magnitude"] = mag_column

The reader function we will use below requires the DataFrame key "time" to define the reference time it is to use to define waveform segments.   For clarity I chose to set the arrival time from the ANF pick table to "Ptime".   Hence, we need this trivial patch from the pandas API.

In [8]:
df = df.rename(columns={"Ptime" : "time"})

In [9]:
# now limit by magnitude 
minimum_magnitude = 5.5
df = df.query(f"magnitude >= {minimum_magnitude}")
print("Size of subsetted DataFrame=",len(df))

Size of subsetted DataFrame= 17336


## Build Parallel Input
The ugly work for all this inside a function define in what is currently a local module called *s3_segment_reader*.   We drive the processing by days as the data are archived as day files.   This little function just generates a list of dictionaries (process_docs) that will drive the actual waveform extraction.

In [10]:
#import importlib
import s3segment_reader as s3sr
#importlib.reload(s3sr)
process_docs = s3sr.run_by_days(df,
                               -240,
                               300,
                               db,
                               auxkeys=["evid","orid","arid","iphase","delta","pick_channel"],
                               )

In [11]:
len(process_docs)

9493

## Run Retrieval Function
First, it will be useful in discussing this to look at the content of one of the document in process_docs.   This the first of thousands.

In [12]:
from mspasspy.util.seismic import print_metadata
# verify doc structure is as expected
print_metadata(process_docs[0])

{
  "net": "AZ",
  "sta": "MONP2",
  "channel_select": "B*",
  "s3objects": [
    "miniseed/AZ/2010/060/MONP2.AZ.2010.060"
  ],
  "arrival": [
    {
      "time": 1267412206.06108,
      "evid": 298936.0,
      "orid": 472058.0,
      "arid": 24260846,
      "iphase": "P",
      "delta": 79.438,
      "pick_channel": "BHZ"
    },
    {
      "time": 1267430467.60053,
      "evid": 298948.0,
      "orid": 472072.0,
      "arid": 24263873,
      "iphase": "P",
      "delta": 78.864,
      "pick_channel": "BHZ"
    }
  ],
  "start": [
    1267411866.06108,
    1267430127.60053
  ],
  "end": [
    1267412606.06108,
    1267430867.60053
  ]
}


This function is needed to allow the data to be saved in parallel.

In [13]:
from mspasspy.util.seismic import number_live
from mspasspy.algorithms.bundle import bundle_seed_data
from obspy import UTCDateTime
def save_segment_outputs(ens,db,outdir="wf_TimeSeries"):
    """
    Completion function to save outputs to database using a single, common file to store all. 
    Default creates this as a raw binary file of a gazillion doubles.   

    Uses a single Database instance (db) as this function will run under this notebook's 
    interpreter.   This could be a bottleneck, but a volatile test below will verify that. 

    Returns a tuple with (ensemble_size, number_live, elog ) as content where elog 
    is the ErrorLogger attribute of ens.   In this context it is rarely empty but 
    content can be used to appraise failures.  
    """
    nlive = number_live(ens)
    nsize = len(ens.member)
    if nlive>0:
        t = UTCDateTime(ens.member[0].t0)
        dfile = f"{t.year}_{t.julday}.dat"
    else:
        # this should never appear as an actual file
        dfile = "BADDATA_DO_NOT_SAVE"
    ens = db.save_data(ens,
                        collection="wf_TimeSeries",
                        storage_mode="file",
                        dir=outdir,
                        dfile=dfile,
                        return_data=True,    # Default throws ens away
                    )
    return (nsize,nlive,ens.elog)
    

This next pair of boxes do most of the work.   It will run for a while.  

In [14]:
from mspasspy.workflow import sliding_window_pipeline

In [15]:
import time
t0=time.time()
print("Submitting retrieve to dask cluster")
out = sliding_window_pipeline(process_docs,
                              s3sr.get_s3_segments,
                              dask_client,
                              completion_function=save_segment_outputs,
                              cfunc_args=[db],
                             #verbose=True,
                             #progress_report_interval=1000,
                             )
t=time.time()
print("Elapsed time=",t-t0)
print("Number of items handled=",len(process_docs))

Submitting retrieve to dask cluster
Elapsed time= 1817.6841714382172
Number of items handled= 9493


In [16]:
# get grand totals 
nhandled=0
nlive=0
for nh,nl,elog in out:
    nhandled += nh
    nlive += nl
print("Retrieved a total of ",nhandled," waveform segments")
print("Number TimeSeries objects saved as live data in the working database=",nlive)

Retrieved a total of  46124  waveform segments
Number TimeSeries objects saved as live data in the working database= 46124


In [17]:
print("List of log of all problems")
for nhm,nl,elog in out:
    #print(elog.size())
    if elog.size()>0:
        print(elog.get_error_log())
    

List of log of all problems


We need to deal with a disconnect between how we created our source collection and the normal MsPASS expectation.  Specifically, we created this data set from css3.0 tables where source data ids are defined by two integer keys called "evid" and "orid".   We kept only "orid" for reasons that are not necessary for this class.   The problem is MsPASS wants to normally use the key "source_id" that is the ObjectId of a document in the source collection.   The following algorithm addresses that in a somewhat obscure way done for efficiency.   That is, it builds the update list and then its the database server all at once with "update_many".   Experience has shown that is orders of magnitude faster than single document transactions. 

In [18]:
# first build a cross-reference dictionary - rational because source is small
orid_xref=dict()
cursor = db.source.find({})
for doc in cursor:
    # type conversion is safer
    orid = int(doc['orid'])
    sid = doc['_id']
    orid_xref[orid] = sid
print("Size of orid cross-reference dictionary=",len(orid_xref))

Size of orid cross-reference dictionary= 342


In [19]:
from pymongo import UpdateOne
from bson.objectid import ObjectId
operations = list()
cursor = db.wf_TimeSeries.find({})
nfail = 0   # just count problems instead of echoing errors
for doc in cursor:
    wfid = doc['_id']
    if "orid" in doc:
        orid = int(doc["orid"])
        if orid in orid_xref:
            sid = orid_xref[orid]
            operations.append(
                UpdateOne(
                    {"_id": ObjectId(wfid)},      # The Filter
                    {"$set": {"source_id": sid}}     # The Update
                )
    )

# Send the entire batch to MongoDB at once
if operations:
    result = db.wf_TimeSeries.bulk_write(operations)

In [20]:

from mspasspy.util.db_utils import MongoDBWorker,fetch_dbhandle
dbplugin = MongoDBWorker(mspass_client)
dask_client.register_plugin(dbplugin)

{'tcp://127.0.0.1:35123': {'status': 'OK'},
 'tcp://127.0.0.1:38635': {'status': 'OK'},
 'tcp://127.0.0.1:39651': {'status': 'OK'},
 'tcp://127.0.0.1:46873': {'status': 'OK'}}

This next section creates a different view of the data as three component bundles we call `Seismogram` objects in MsPASS.

Noted added for final push to github.  This code actually doesn't do what I intended but works fine.  I intended to have data file written to "./wf_Seismogram/ES26raw_seismograms.dat".   Instead it writes to "~/wf_Seismogram/ES26raw_seismogram.dat".   Makes cleanup a bit more confusing but it works fine.  A classic example of a file management problem.  

In [21]:
from mspasspy.db.normalize import MiniseedMatcher,normalize
from mspasspy.algorithms.bundle import bundle_seed_data
from mspasspy.util.seismic import number_live,print_metadata

def run_bundle(query,dbname,channel_matcher=None,site_matcher=None):
    db = fetch_dbhandle(dbname)
    # the default is very slow as every ensemble will have to do this
    if channel_matcher is None:
        channel_matcher=MiniseedMatcher(db,attributes_to_load=['starttime', 'endtime', 'lat', 'lon', 'elev', '_id','hang','vang'])
    if site_matcher is None:
        site_matcher = MiniseedMatcher(db,
                                       collection='site',
                                       attributes_to_load=['starttime', 'endtime', 'lat', 'lon', 'elev', '_id'],
                                      )
        
    cursor = db.wf_TimeSeries.find(query)
    tsens = db.read_data(cursor,collection="wf_TimeSeries")
    #print("ts ensemble size=",len(tsens.member))
    tsens = normalize(tsens,channel_matcher,handles_ensembles=False)
    #print(tsens.live)
    # this fixes an old bug from the old version being run here
    tsens.set_live()
    #print("Number live after normalize=",number_live(tsens))
    #print_metadata(tsens.member[0])
    ens = bundle_seed_data(tsens)
    del tsens
    #print("bundle output size=",len(ens.member))
    #print("ensemble live=",ens.live)
    #print(ens.elog.get_error_log())
    ens = normalize(ens,site_matcher,handles_ensembles=False)
    # this old version had a bug in normalize that was fixed 
    # this is a patch until a new version can be installed on geolab
    ens.set_live()
    #print("After normalize live=",ens.live)
    #print("elog size=",ens.elog.size())
    #print(ens.elog.get_error_log())
    ntotal = len(ens.member)
    nlive = number_live(ens)
    #for d in ens.member:
    #    print(d.live,d.elog.size())
    #print_metadata(ens.member[0])
    ret=db.save_data(ens,
                     storage_mode="file",
                     collection="wf_Seismogram", 
                     dir="wf_Seismogram",
                     dfile="ES26raw_seismograms.dat",
                    )
    return ntotal,nlive

This builds a cross-reference index that is preferable to constant matching of miniseed net codes.

In [22]:
channel_matcher=MiniseedMatcher(db,attributes_to_load=['starttime', 'endtime', 'lat', 'lon', 'elev', '_id','hang','vang'])
site_matcher = MiniseedMatcher(db,
                                collection='site',
                                attributes_to_load=['starttime', 'endtime', 'lat', 'lon', 'elev', '_id'],
                            )
        

In [23]:
from mspasspy.algorithms.bundle import bundle_seed_data
# extract a list of source id values that actually have data
srcidlist = db.wf_TimeSeries.distinct("source_id")
# process by ensemble 
# a subtle feature of MsPASS is default storage of ensembles are faster to read than atomic processing
querylist=list()
for srcid in srcidlist:
    query={"source_id" : srcid}
    querylist.append(query)
print("Number of ensembles to be processed = ",len(querylist))

Number of ensembles to be processed =  48


In [24]:
t0=time.time()
out = sliding_window_pipeline(querylist,
                              run_bundle,
                              dask_client,
                              pfunc_args=[db.name,channel_matcher,site_matcher],
                              #verbose=True,
                             #progress_report_interval=1000,
                             )
t=time.time()
print("Time to process with run_bundle=",t-t0)

Time to process with run_bundle= 207.58718824386597


Let's quickly validate the size of the data set we just created.

In [25]:
nts=db.wf_TimeSeries.count_documents({})
print("This workflow generated ",nts," TimeSeries (single channel) waveform segments")
nseis=db.wf_Seismogram.count_documents({})
print("This workflow generated ",nseis," Seismogram (three component) waveform segments")
print("Number lost in conversion = ",nts-3*nseis)

This workflow generated  46124  TimeSeries (single channel) waveform segments
This workflow generated  15102  Seismogram (three component) waveform segments
Number lost in conversion =  818


Finally, we need this index for efficiency in the data processing we will be doing in Session 2. 

In [26]:
db.wf_Seismogram.create_index("source_id")

'source_id_1'